**1. Import Libraries and Mount to Drive**

In [ ]:
# install required packages
!pip install evaluate -q
!pip install --upgrade torchao -q

# import libraries
import zipfile
import os
import torch
import re
import evaluate
import random
import math
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset

# mount to personal Google Drive
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 60.5 MB/s eta 0:00:00


Mounted at /content/drive


**2. Load Full Model Zip**

In [ ]:
# path to the model zip file in google drive
zip_path = "/content/drive/MyDrive/Fine-Tuning-Notebooks/Full-Fine-Tuning/Llama-3.2-3B-Instruct/Medical/medical_llama3_full_model.zip"

# where to unzip the model in colab
output_path = "/content/model"

# makes the output folder if it does not already exist
os.makedirs(output_path, exist_ok=True)

# unzips the model into the colab workspace
print("Unzipping model...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(output_path)

# confirms the model has been unzipped and shows the folder contents
print(f"Model unzipped to {output_path}")
print(os.listdir(output_path))

Unzipping model...
Model unzipped to /content/model
['medical_llama3_full_model']


**3. Load Model Llama-3.2-3B-Instruct (Medical Full)**


In [ ]:
# the model name
model_name = "llama3_medical_full"

# path to the unzipped full fine-tuned model
model_path = "/content/model/medical_llama3_full_model"

# loads the tokenizer from the full model folder
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path)

# add padding token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# loads the full fine-tuned model directly
print("Loading full fine-tuned model...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    device_map="auto"
)

model.eval()

print("Model loaded and ready!")

# confirm model loaded correctly
print(model.config._name_or_path)

Loading tokenizer...
Loading full fine-tuned model...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Model loaded and ready!
/content/model/medical_llama3_full_model


**4. Load compute_overall_accuracy**

In [ ]:
# official accuracy computation function from medcalc-bench (table_stats.py)
def compute_overall_accuracy(output_path, model_name, prompt_style):
    category_accuracy = {}

    with open(f"outputs/{output_path}") as file:
        for line in file:
            data = json.loads(line)

            category = data["Category"]

            if category not in category_accuracy:
                category_accuracy[category] = []

            if data["Result"] == "Correct":
                category_accuracy[category].append(1)
            else:
                category_accuracy[category].append(0)

    # compute average and standard deviation for each category
    category_stats = {}
    all_results = []

    for cat, results in category_accuracy.items():
        results_array = np.array(results)
        category_mean = np.mean(results_array)
        category_std = round(np.sqrt(category_mean * (1-category_mean) / len(results_array)), 2)
        category_stats[cat] = {
            "average": round(category_mean * 100, 2),
            "std": category_std
        }
        all_results.extend(results)

    # compute overall average and standard deviation
    all_results_array = np.array(all_results)
    overall_average = np.mean(all_results_array)
    overall_std =  round(np.sqrt(overall_average * (1-overall_average) / 1047), 2)

    category_stats["overall"] = {
        "average": round(overall_average * 100, 2),
        "std": overall_std
    }

    if not os.path.exists("results"):
        os.makedirs("results")

    if "/" in model_name:
        model_name = model_name.split('/')[1]

    with open(f"results/results_{model_name}_{prompt_style}.json", "w") as file:
        json.dump(category_stats, file, indent=4)

    return category_stats

**5. Perform Evaluation (MedCalc-Bench-v1.0)**

In [ ]:
# set device to gpu if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# set seed for reproducibility
torch.manual_seed(42)

# official zero shot cot prompt from medcalc-bench
def zero_shot(note, question):
    system_msg = 'You are a helpful assistant for calculating a score for a given patient note. Please think step-by-step to solve the question and then generate the required score. Your output should only contain a JSON dict formatted as {"step_by_step_thinking": str(your_step_by_step_thinking_procress_to_solve_the_question), "answer": str(short_and_direct_answer_of_the_question)}.'
    user_temp = f'Here is the patient note:\n{note}\n\nHere is the task:\n{question}\n\nPlease directly output the JSON dict formatted as {{"step_by_step_thinking": str(your_step_by_step_thinking_procress_to_solve_the_question), "answer": str(short_and_direct_answer_of_the_question)}}:'
    return system_msg, user_temp

# official extract answer function from medcalc-bench
def extract_answer(answer, calid):
    calid = int(calid)
    extracted_answer = re.findall(r'[Aa]nswer":\s*(.*?)\}', answer)

    matches = re.findall(r'"step_by_step_thinking":\s*"([^"]+)"\s*,\s*"[Aa]nswer"', answer)
    if matches:
        explanation = matches[-1]
    else:
        explanation = "No Explanation"

    if len(extracted_answer) == 0:
        extracted_answer = "Not Found"
    else:
        extracted_answer = extracted_answer[-1].strip().strip('"')
        if extracted_answer in [
            "str(short_and_direct_answer_of_the_question)",
            "str(value which is the answer to the question)",
            "X.XX"
        ]:
            extracted_answer = "Not Found"

    if calid in [13, 68]:
        match = re.search(r"^(0?[1-9]|1[0-2])\/(0?[1-9]|[12][0-9]|3[01])\/(\d{4})", extracted_answer)
        if match:
            month = int(match.group(1))
            day = int(match.group(2))
            year = match.group(3)
            answer = f"{month:02}/{day:02}/{year}"
        else:
            answer = "N/A"

    elif calid in [69]:
        extracted_answer = extracted_answer.replace("[", "(").replace("]", ")").replace("'", "").replace('"', "")
        match = re.search(r"\(?[\"\']?(\d+)\s*(weeks?)?[\"\']?,?\s*[\"\']?(\d+)\s*(days?)?[\"\']?\s*\)?", extracted_answer)
        if match:
            weeks = match.group(1)
            days = match.group(3)
            answer = f"({weeks}, {days})"
        else:
            answer = "N/A"

    elif calid in [4, 15, 16, 17, 18, 20, 21, 25, 27, 28, 29, 32, 33, 36, 43, 45, 48, 51]:
        match = re.search(r"(\d+) out of", extracted_answer)
        if match:
            answer = match.group(1)
        else:
            match = re.search(r"-?\d+(, ?-?\d+)+", extracted_answer)
            if match:
                answer = str(len(match.group(0).split(",")))
            else:
                match = re.findall(r"(-?\d+(\.\d+)?)", extracted_answer)
                if len(match) > 0:
                    answer = match[-1][0]
                else:
                    answer = "N/A"

    elif calid in [2, 3, 5, 6, 7, 8, 9, 10, 11, 19, 22, 23, 24, 26, 30, 31, 38, 39, 40, 44, 46, 49, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67]:
        match = re.search(r"str\((.*)\)", extracted_answer)
        if match:
            expression = match.group(1).replace("^", "**").replace("is odd", "% 2 == 1").replace("is even", "% 2 == 0").replace("sqrt", "math.sqrt").replace(".math", "").replace("weight", "").replace("height", "").replace("mg/dl", "").replace("g/dl", "").replace("mmol/L", "").replace("kg", "").replace("g", "").replace("mEq/L", "")
            expression = expression.split('#')[0]
            if expression.count('(') > expression.count(')'):
                expression += ')' * (expression.count('(') - expression.count(')'))
            elif expression.count(')') > expression.count('('):
                expression = '(' * (expression.count(')') - expression.count('(')) + expression
            try:
                answer = eval(expression, {"__builtins__": None}, {"min": min, "pow": pow, "round": round, "abs": abs, "int": int, "float": float, "math": math, "np": np, "numpy": np})
            except:
                answer = "N/A"
        else:
            match = re.search(r"(-?\d+(\.\d+)?)\s*mL/min/1.73", extracted_answer)
            if match:
                answer = eval(match.group(1))
            else:
                match = re.findall(r"(-?\d+(\.\d+)?)\%", extracted_answer)
                if len(match) > 0:
                    answer = eval(match[-1][0]) / 100
                else:
                    match = re.findall(r"(-?\d+(\.\d+)?)", extracted_answer)
                    if len(match) > 0:
                        answer = eval(match[-1][0])
                    else:
                        answer = "N/A"
        if answer != "N/A":
            answer = str(answer)
    else:
        answer = "N/A"

    return answer, explanation

# official correctness check from medcalc-bench
def check_correctness(answer, ground_truth, calid, upper_limit, lower_limit):
    calid = int(calid)
    try:
        if calid in [13, 68]:
            if datetime.strptime(answer, "%m/%d/%Y").strftime("%-m/%-d/%Y") == datetime.strptime(ground_truth, "%m/%d/%Y").strftime("%-m/%-d/%Y"):
                correctness = 1
            else:
                correctness = 0

        elif calid in [69]:
            match = re.search(r"\(?[\"\']?(\d+)\s*(weeks?)?[\"\']?,?\s*[\"\']?(\d+)\s*(days?)?[\"\']?\s*\)?", ground_truth)
            ground_truth = f"({match.group(1)}, {match.group(3)})"
            match = re.search(r"\(?[\"\']?(\d+)\s*(weeks?)?[\"\']?,?\s*[\"\']?(\d+)\s*(days?)?[\"\']?\s*\)?", answer)
            if match:
                weeks = match.group(1)
                days = match.group(3)
                answer = f"({weeks}, {days})"
                correctness = 1 if eval(answer) == eval(ground_truth) else 0
            else:
                correctness = 0

        elif calid in [4, 15, 16, 17, 18, 20, 21, 25, 27, 28, 29, 32, 33, 36, 43, 45, 48, 51]:
            answer = round(eval(answer))
            correctness = 1 if answer == eval(ground_truth) else 0

        elif calid in [2, 3, 5, 6, 7, 8, 9, 10, 11, 19, 22, 23, 24, 26, 30, 31, 38, 39, 40, 44, 46, 49, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67]:
            answer = eval(answer)
            correctness = 1 if eval(lower_limit) <= answer <= eval(upper_limit) else 0

        else:
            return None

    except Exception:
        return None

    return correctness

# load the official test set
df = pd.read_csv("test_data.csv")
test_set = df

print(f"Running evaluation on all {len(test_set)} questions...")

correct_count = 0
wrong_count = 0
skipped_count = 0
total = len(test_set)

category_stats = {}

output_path = f"{model_name.replace('/', '_')}_zero_shot.jsonl"

if not os.path.exists("outputs"):
    os.makedirs("outputs")

# set eos tokens based on model family
if "llama" in model_name.lower():
    eos_token_ids = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")]
elif "qwen" in model_name.lower():
    eos_token_ids = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")]
else:
    eos_token_ids = [tokenizer.eos_token_id]

# official loop from medcalc-bench
for index in tqdm(range(len(test_set))):
    example = test_set.iloc[index]

    system, user = zero_shot(example["Patient Note"], example["Question"])

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    input_length = inputs["input_ids"].shape[1]

    # official-style generation limit using the model's actual context length
    model_context_length = getattr(model.config, "max_position_embeddings", tokenizer.model_max_length)

    # fallback if tokenizer has an unrealistic huge placeholder value
    if model_context_length is None or model_context_length > 1_000_000:
        model_context_length = tokenizer.model_max_length

    max_new_tokens = min(4096, max(1, model_context_length - input_length))

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=eos_token_ids,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    response = re.sub(r"\s+", " ", response)

    try:
        predicted, explanation = extract_answer(response, example['Calculator ID'])
    except Exception as e:
        predicted = "N/A"
        explanation = str(e)

    if predicted not in ["N/A", "Not Found", None]:
        try:
            correctness = check_correctness(
                answer=predicted,
                ground_truth=example['Ground Truth Answer'],
                calid=example['Calculator ID'],
                upper_limit=example['Upper Limit'],
                lower_limit=example['Lower Limit']
            )
        except Exception:
            correctness = None
    else:
        correctness = None

    if correctness is None:
        skipped_count += 1
        result_str = "Skipped"
    elif correctness == 1:
        correct_count += 1
        result_str = "Correct"
    else:
        wrong_count += 1
        result_str = "Wrong"

    cat = example.get('Category', 'Unknown')
    if cat not in category_stats:
        category_stats[cat] = {'correct': 0, 'wrong': 0, 'skipped': 0}
    if correctness is None:
        category_stats[cat]['skipped'] += 1
    elif correctness == 1:
        category_stats[cat]['correct'] += 1
    else:
        category_stats[cat]['wrong'] += 1

    print(f"\n{'-' * 50}")
    print(f"Question {index+1} of {total}")
    print(f"Calculator:       {example['Calculator Name']}")
    print(f"Output type:      {example['Output Type']}")
    print(f"Ground truth:     {example['Ground Truth Answer']}")
    print(f"Acceptable range: {example['Lower Limit']} - {example['Upper Limit']}")
    print(f"Model response:   {response}")
    print(f"Extracted answer: {predicted}")
    print(f"Result:           {result_str}")
    print(f"{'-' * 50}")

    with open(f"outputs/{output_path}", "a") as f:
        f.write(json.dumps({
            "Row Number": int(example["Row Number"]),
            "Calculator Name": example['Calculator Name'],
            "Calculator ID": str(example['Calculator ID']),
            "Category": example['Category'],
            "Note ID": str(example['Note ID']),
            "Patient Note": example['Patient Note'],
            "Question": example['Question'],
            "LLM Answer": predicted,
            "LLM Explanation": explanation,
            "Ground Truth Answer": example['Ground Truth Answer'],
            "Ground Truth Explanation": example['Ground Truth Explanation'],
            "Result": "Correct" if correctness == 1 else "Incorrect"
        }) + "\n")

# calculates the final stats
answered = correct_count + wrong_count
coverage = (answered / total * 100)
accuracy_all = (correct_count / total * 100)

print("\n" + "=" * 50)
print("EVALUATION COMPLETE")
print(f"Total questions:          {total}")
print(f"Correct answers:          {correct_count}")
print(f"Wrong answers:            {wrong_count}")
print(f"Skipped/Invalid:          {skipped_count}")
print(f"Coverage:                 {coverage:.2f}%")
print(f"Accuracy (all {total}):   {accuracy_all:.2f}%")
print("=" * 50)

# official accuracy checker
compute_overall_accuracy(output_path, model_name, "zero_shot")

Running evaluation on all 1047 questions...


  0%|          | 1/1047 [00:11<3:13:00, 11.07s/it]


--------------------------------------------------
Question 1 of 1047
Calculator:       Creatinine Clearance (Cockcroft-Gault Equation)
Output type:      decimal
Ground truth:     25.238
Acceptable range: 23.9761 - 26.4999
Model response:   {"step_by_step_thinking": "To calculate the patient's Creatinine Clearance using the Cockroft-Gault Equation, we need to first determine the patient's adjusted body weight in kg. The patient's BMI is not provided, so we will assume that the patient is not overweight or obese.\n\nNext, we will use the Cockroft-Gault Equation to calculate the creatinine clearance rate:\n\nCreatinine Clearance (mL/min) = (140 - age) * (weight in kg) * (0.85 if female) / (72 * serum creatinine (mg/dL) * 24)\n\nGiven that the patient is a male, we will use the formula without the gender adjustment factor.\n\nNow, let's substitute the given values into the equation:\n\nCreatinine Clearance (mL/min) = (140 - 87) * 48 kg / (72 * 1.73 mg/dL * 24)\n\nCalculating the equation

  0%|          | 2/1047 [00:19<2:42:10,  9.31s/it]


--------------------------------------------------
Question 2 of 1047
Calculator:       Creatinine Clearance (Cockcroft-Gault Equation)
Output type:      decimal
Ground truth:     38
Acceptable range: 36.1 - 39.9
Model response:   {"step_by_step_thinking": "To calculate the patient's Creatinine Clearance using the Cockroft-Gault Equation, we need to know the patient's serum creatinine level, age, sex, weight, and urine output. The formula for the Cockroft-Gault Equation is: \nCreatinine Clearance (mL/min) = (140 - age) * (weight in kg) * (0.85 if female) / (72 * serum creatinine) * urine output (mL/min). \nBased on the patient's information, we know that the patient is a 83-year-old male with a weight of 67.2 kg. \nSince the patient's BMI is not provided, we will assume that the patient is not overweight or obese and use their actual weight.\nWe also know that the patient's serum creatinine level is not provided, so we cannot calculate the creatinine clearance directly. \nHowever, we 

...

100%|█████████▉| 1046/1047 [2:44:31<00:06,  6.41s/it]


--------------------------------------------------
Question 1046 of 1047
Calculator:       Estimated Gestational Age
Output type:      integer
Ground truth:     ('34 weeks', '6 days')
Acceptable range: ('34 weeks', '6 days') - ('34 weeks', '6 days')
Model response:   {"step_by_step_thinking": "To calculate the gestational age, we need to calculate the difference between the current date and the last menstrual period.\nWe know that the last menstrual period was on 07/18/2013.\nToday's date is 03/19/2014.\nTo calculate the difference, we need to convert both dates to the same format.\nLet's convert both dates to the Gregorian calendar format (YYYY-MM-DD).\nThe last menstrual period was on 07/18/2013 which is equivalent to 07/18/2013.\nToday's date is 03/19/2014.\nNow we can calculate the difference between the two dates.\nThe difference is 1 year and 8 months.\nSince the gestational age is calculated from the first day of the last menstrual period, we need to add 28 days to the differen

100%|██████████| 1047/1047 [2:44:38<00:00,  9.43s/it]


--------------------------------------------------
Question 1047 of 1047
Calculator:       Estimated Gestational Age
Output type:      integer
Ground truth:     ('14 weeks', '2 days')
Acceptable range: ('14 weeks', '2 days') - ('14 weeks', '2 days')
Model response:   {"step_by_step_thinking": "To calculate the gestational age, we need to calculate the difference between the current date and the last menstrual period.\nWe know that the last menstrual period was on 10/16/2007 and today's date is 01/24/2008.\nWe can calculate the difference by subtracting the day of the last menstrual period from the current day.\nSince the current date is 01/24/2008 and the last menstrual period was on 10/16/2007, we can calculate the difference as follows:\n24 - 16 = 8\nSince there are 7 days in a week, we can divide the difference by 7 to get the number of weeks.\n8 / 7 = 1 with a remainder of 1.\nTherefore, the patient's estimated gestational age is (1 week, 1 day).", "answer": "(1 week, 1 day)"}
Ext

{'lab': {'average': np.float64(7.65), 'std': np.float64(0.01)},
 'risk': {'average': np.float64(6.67), 'std': np.float64(0.02)},
 'physical': {'average': np.float64(10.42), 'std': np.float64(0.02)},
 'severity': {'average': np.float64(7.5), 'std': np.float64(0.03)},
 'diagnosis': {'average': np.float64(15.0), 'std': np.float64(0.05)},
 'date': {'average': np.float64(6.67), 'std': np.float64(0.03)},
 'dosage': {'average': np.float64(2.5), 'std': np.float64(0.02)},
 'overall': {'average': np.float64(8.21), 'std': np.float64(0.01)}}